In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

df = pd.read_csv('../pump_anomaly_detection/data/training/features_scaled.csv')

X = df.drop(columns=['file'])
# flow_dom_freq is identical to dominant_freq_hz (r=1.0, redundant)
# flow_dom_freq_global has extreme outlier values and is uninformative
X = X.drop(columns=['flow_dom_freq', 'flow_dom_freq_global'])

corr = X.corr()
n = len(X.columns)

label_map = {
    'dominant_freq_hz':            'Dominant Frequency (Hz)',
    'spectral_power_in_band':      'Spectral Band Power',
    'spectral_snr':                'Spectral SNR',
    'spectral_entropy':            'Spectral Entropy',
    'freq_stability_std':          'Frequency Stability',
    'interval_cv':                 'Stroke Interval CV',
    'stroke_amp_cv':               'Stroke Amplitude CV',
    'rise_fall_ratio':             'Rise/Fall Ratio',
    'rise_time_cv':                'Rise Time CV',
    'fall_time_cv':                'Fall Time CV',
    'autocorr_at_period':          'Autocorr. at Period',
    'cycle_shape_cv':              'Cycle Shape CV',
    'cycle_corr_mean':             'Cycle Correlation',
    'phase_portrait_eccentricity': 'Phase Eccentricity',
    'envelope_cv':                 'Envelope CV',
    'top_bot_corr':                'Top-Bottom Corr.',
    'left_right_corr':             'Left-Right Corr.',
    'quadrant_cv_max':             'Quadrant CV (Max)',
    'active_roi_fraction':         'Active ROI Fraction',
    'motion_cv':                   'Motion CV',
}
labels = [label_map.get(c, c) for c in X.columns]

mask = np.triu(np.ones((n, n), dtype=bool), k=1)
plot_data = corr.values.copy()
plot_data[mask] = np.nan

fig, ax = plt.subplots(figsize=(13, 12))

cmap = plt.cm.RdBu_r.copy()
cmap.set_bad('white')
im = ax.imshow(plot_data, cmap=cmap, vmin=-1, vmax=1, aspect='auto')

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(labels, fontsize=9)
ax.tick_params(top=False, bottom=True, labeltop=False, labelbottom=True)

for i in range(n):
    for j in range(i + 1):
        val = corr.values[i, j]
        color = 'white' if abs(val) > 0.65 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=5.5, color=color)

for i in range(n):
    for j in range(i):
        if abs(corr.values[i, j]) > 0.97:
            rect = patches.Rectangle(
                (j - 0.5, i - 0.5), 1, 1,
                linewidth=2, edgecolor='limegreen', facecolor='none'
            )
            ax.add_patch(rect)

cb = plt.colorbar(im, ax=ax, label='Pearson r', shrink=0.75, pad=0.02)
cb.ax.tick_params(labelsize=9)

plt.tight_layout()
plt.savefig('corr_heatmap.pdf', bbox_inches='tight', dpi=150)
plt.savefig('corr_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
